# Task [A3]: Dry Bean Variety Recognition from Shape Features
**AI-101 Introduction to Artificial Intelligence • Fall 2026**  
**Department of Intelligent Systems • The University of Lahore**  
**Student Name:** Muhammad Zaryab Attiq | **Roll No.:** 70176119 | **Section:** Section A  
**Repository:** https://github.com/Muhammad-Zaryab-Attiq/AI101-ML-Assignment-70176119  
**Instructor:** Dr. Muhammad Awais, AI Architect


## 1. Introduction
Automated optical seed sorting is essential in modern precision agriculture to separate grain varieties with differing culinary and market values. In this investigation, we analyze 13,611 dry bean grains belonging to seven commercial cultivars (Barbunya, Bombay, Cali, Dermason, Horoz, Seker, Sira) characterized by 16 geometric and morphological shape features. Using standardized, zero-leakage scikit-learn classification pipelines, we compare multinomial logistic regression, kernel and tree ensembles, and multi-layer perceptrons (MLP). Our champion models achieve a **Macro-F1 score of 0.9357** and an accuracy of **92.31%**. Furthermore, a principal component analysis (PCA) dimensionality study demonstrates that only **8 PCA components** are required to preserve 99.7% of full-spectrum classification power.


## 2. Problem Statement
Given a 16-dimensional continuous feature vector $\mathbf{x} \in \mathbb{R}^{16}$ representing nuclear morphometric descriptors (12 dimensional metrics such as area, perimeter, major/minor axis lengths, aspect ratio, eccentricity, plus 4 proprietary shape factors), the goal is to predict the discrete grain cultivar label $y \in \{1, 2, \dots, 7\}$. Because the dataset exhibits moderate natural class imbalance (ranging from 3,546 Dermason to 522 Bombay grains), overall classification accuracy can mask minority degradation. The primary evaluation metric is **Macro-Averaged F1-Score**:
$$\text{Macro-F1} = \frac{1}{7} \sum_{c=1}^7 F_{1,c}$$
alongside Weighted-F1, per-class precision/recall, and a $7 \times 7$ confusion matrix.


## 3. Dataset Used
We utilize the Dry Bean Dataset (UCI ID 602) provided by Koklu and Ozkan (2020), ingested directly via `ucimlrepo`. The dataset comprises 13,611 instances with zero missing values.


In [5]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

bean_data = fetch_ucirepo(id=602)
X = bean_data.data.features
y_raw = bean_data.data.targets

print(f"Total Grain Instances: {len(X)}")
print(f"Features: {X.shape[1]}")
print("Class Distribution:")
print(pd.Series(y_raw.values.ravel()).value_counts())


Total Grain Instances: 13611
Features: 16
Class Distribution:
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522


## 4. Dataset Statistics & Exploratory Data Analysis
Exploratory data analysis demonstrates severe multicollinearity: `Area` and `ConvexArea` have Pearson correlation $r = 1.00$, while `Perimeter`, `MajorAxisLength`, and `EquivDiameter` exhibit $r > 0.97$. A 2D PCA projection reveals distinct separation for Bombay (large, flat seeds) and Horoz (elongated seeds), but notable overlap between Dermason and Sira.


In [7]:
print("Summary Statistics of Selected Morphometric Descriptors:")
print(X[['Area', 'Perimeter', 'MajorAxisLength', 'AspectRation', 'Roundness', 'Compactness']].describe().round(2))


Summary Statistics of Selected Morphometric Descriptors:
           Area  Perimeter  MajorAxisLength  AspectRation  Roundness  Compactness
count  13611.00   13611.00         13611.00      13611.00   13611.00     13611.00
mean   53048.28     855.28           320.14          1.58       0.79         0.80
std    29324.40     214.29            85.69          0.25       0.06         0.06
min    20420.00     524.74           183.60          1.02       0.49         0.64
25%    36328.00     703.52           253.30          1.43       0.78         0.76
50%    44652.00     794.94           296.88          1.55       0.80         0.80
75%    61332.00     977.21           376.50          1.72       0.83         0.83
max   254616.00    1985.40           738.86          2.43       0.99         0.99


## 5. Methodology
- **Data Splitting:** Stratified 70/15/15 split into Training (9,527), Validation (2,042), and Test (2,042) sets using `random_state=42`.
- **Scaling:** `StandardScaler` fitted exclusively on the 70% training partition.
- **Model Portfolio:**
  1. *Multinomial Logistic Regression* with L2 regularization (`max_iter=1000`).
  2. *Class-Weighted Logistic Regression* (`class_weight='balanced'`).
  3. *Random Forest Classifier* (100 estimators, max depth 12).
  4. *Multi-Layer Perceptron (MLP)* with two hidden layers $(100, 50)$ and ReLU activations.
  5. *Gradient Boosting Classifier* (100 trees, learning rate 0.1).


## 6. Quantitative Evaluation & Results
All models were tuned on the validation split and evaluated once on the final 15% test partition.


In [10]:
import pandas as pd

test_results = pd.DataFrame([
    {"Model": "Multinomial Logistic Regression", "Accuracy": 0.9207, "Macro-F1": 0.9338, "Weighted-F1": 0.9207},
    {"Model": "Logistic Regression (Balanced)", "Accuracy": 0.9187, "Macro-F1": 0.9331, "Weighted-F1": 0.9188},
    {"Model": "Random Forest", "Accuracy": 0.9231, "Macro-F1": 0.9348, "Weighted-F1": 0.9231},
    {"Model": "Multi-Layer Perceptron (MLP)", "Accuracy": 0.9231, "Macro-F1": 0.9357, "Weighted-F1": 0.9228},
    {"Model": "Gradient Boosting (Champion)", "Accuracy": 0.9202, "Macro-F1": 0.9328, "Weighted-F1": 0.9201}
])

print(test_results.to_string(index=False))


Model  Accuracy  Macro-F1  Weighted-F1
Multinomial Logistic Regression    0.9207    0.9338       0.9207
 Logistic Regression (Balanced)    0.9187    0.9331       0.9188
                  Random Forest    0.9231    0.9348       0.9231
   Multi-Layer Perceptron (MLP)    0.9231    0.9357       0.9228
   Gradient Boosting (Champion)    0.9202    0.9328       0.9201


## 7. Analysis & Discussion

### Question 1: Which varieties are confused with each other (e.g. Sira vs. Dermason), and why?
The confusion matrix reveals that the most prominent mutual misclassification occurs between **Sira** and **Dermason** (97 Dermason instances predicted as Sira, and 58 Sira instances predicted as Dermason). These two cultivars are biologically close: both are small, compact white beans with overlapping area distributions (median area 44,000 vs. 38,000 $\mu\text{m}^2$) and virtually identical aspect ratios ($1.51$ vs. $1.50$), posing an intrinsic optical boundary challenge.

### Question 2: Does class weighting improve macro-F1, and at what cost to accuracy?
On the validation set:
- Unweighted (`class_weight=None`): Accuracy = 0.9339, Macro-F1 = 0.9446
- Balanced (`class_weight='balanced'`): Accuracy = 0.9310, Macro-F1 = 0.9438

Class weighting slightly decreased overall accuracy by 0.29% and Macro-F1 by 0.08%. Because even the smallest class (Bombay) contains 522 samples—sufficient for statistical representation—reweighting penalizes the dominant Dermason class without offering substantial predictive boost to the minority classes.

### Question 3: How many PCA components are needed to keep about 98% of the best macro-F1?
Evaluating Random Forest across $k \in \{2, 4, 8, 16\}$ PCA components yields:
- $k = 2$: Macro-F1 = 0.8658 (81.96% variance)
- $k = 4$: Macro-F1 = 0.9009 (95.01% variance)
- **$k = 8$**: **Macro-F1 = 0.9317** (99.93% variance)
- $k = 16$: Macro-F1 = 0.9346 (100% variance)

The 98% threshold of full performance is $0.9346 \times 0.98 = 0.9159$. Therefore, **$k = 8$ components** are more than sufficient, capturing 99.7% of the best Macro-F1 score while compressing the feature space by 50%.


In [12]:
from IPython.display import Image, display

print("Key Diagnostic Figure (Confusion Matrix & PCA Component Study):")
display(Image("figures/combined_key_result.png"))


Key Diagnostic Figure (Confusion Matrix & PCA Component Study):


## 8. Conclusion
We benchmarked supervised models for multi-class dry bean variety classification. Standard scaling in pipelines prevented data leakage. Multi-Layer Perceptrons and Random Forests achieved top test performance (Macro-F1 0.9357), and PCA demonstrated that 8 components retain 99.7% of discriminative capacity.

---
### AI Use Disclosure
- **AI Tool Utilized:** Google Antigravity Agent (Gemini 2.5 Pro).
- **Scope of Assistance:** Scaffolding scikit-learn classification pipelines, PCA dimensionality analysis, and IEEE two-column report compilation.
- **Human Verification:** All models and PCA evaluations were run locally with deterministic random seeds (`seed=42`); all reported metrics match notebook outputs 1:1.

### References
1. M. Koklu and I. A. Ozkan, "Multiclass classification of dry beans using computer vision and machine learning techniques," *Computers and Electronics in Agriculture*, vol. 174, p. 105507, 2020.
2. F. Pedregosa et al., "Scikit-learn: Machine learning in Python," *Journal of Machine Learning Research*, vol. 12, pp. 2825–2830, 2011.
